# 09 · E4: optical dimension (Figure 3)

**Runtime:** T4 GPU. **Time:** about 20–40 minutes. **Resumable.** Needs notebook 08 (decision D-E3).

**Question.** Does a larger optical feature space improve learning, and does it saturate where digital
random features do?

**Design** (protocol E4, Core). D/d ∈ {0.5, 1, 2, 4} at the frozen placement D-E3. Maps: F2-scatter,
F2-fourier, H, C, B (A as reference). N = 100 and full data, 3 map seeds. **D = 8d is added automatically
only if an optical map still gains more than 1 pp from 2d to 4d** (rule fixed in advance). For D > 4096,
F-fourier reads a 128 × 128 far-field grid instead of 64 × 64.

**Interpretation (protocol).** If the optical map saturates at a lower D than H, it is more feature-efficient
(fewer camera pixels needed); if both saturate together, they have equivalent capacity.

In [ ]:
# ---- Setup (edit the two lines below once) ------------------------------------------
GITHUB_REPO = "surensagh/PhotonicsAI-Project"                    # repo that holds this code
WORKSPACE   = "/content/drive/MyDrive/optical_lm_workspace"      # splits, cache, runs, reports
# -------------------------------------------------------------------------------------
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(WORKSPACE, exist_ok=True)
os.environ["OPTLM_WORKSPACE"] = WORKSPACE
os.environ["HF_HOME"] = "/content/hf_cache"          # downloads stay on the fast local disk

# Private repo: add a GitHub token as a Colab secret named GH_TOKEN (key icon in the left bar).
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{GITHUB_REPO}.git" if token else f"https://github.com/{GITHUB_REPO}.git"
REPO_DIR = "/content/" + GITHUB_REPO.split("/")[-1]
cmd = ["git", "clone", "-q", url, REPO_DIR] if not os.path.isdir(REPO_DIR) else ["git", "-C", REPO_DIR, "pull", "-q", "--ff-only"]
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    msg = res.stderr.replace(token, "***") if token else res.stderr
    raise RuntimeError("git failed (check GITHUB_REPO / GH_TOKEN):\n" + msg)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("repo   :", REPO_DIR)
print("commit :", subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip())
print("dirty  :", bool(subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"]).strip()))
print("workspace:", WORKSPACE)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
import json
print(json.load(open(os.path.join(WORKSPACE, "reports", "e3_decision.json"))))

In [ ]:
!python scripts/run_e2.py --config configs/e4.yaml

In [ ]:
print(open(os.path.join(WORKSPACE, "reports", "e4_table.md"), encoding="utf-8").read())

In [ ]:
from IPython.display import Image, display
display(Image(os.path.join(WORKSPACE, "reports", "e4_fig3.png")))